<a href="https://colab.research.google.com/github/lovkush7/NLP--P/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [2]:
path = "/content/gdrive/MyDrive/nlp_news_category_practice.csv"

In [3]:
import pandas as pd
df = pd.read_csv(path)

In [4]:
df.head()

,text,category
0,A security report explains how hackers can exp...,technology
1,"According to a recent report, the hospital int...",health
2,The clinic is offering free health screening t...,health
3,Engineers released an open source project to h...,technology
4,The retailer plans to expand its operations in...,business


In [5]:
df.shape

(720, 2)

In [6]:
df.duplicated().sum()

np.int64(320)

In [7]:
df.drop_duplicates(inplace=True)


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.isnull().sum()

,0
text,0
category,0


In [10]:
import re
def  clean_text(text):
  text = text.lower()
      # Remove URLs
    # Remove URLs
  text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
  text = re.sub(r"<.*?>", "", text)

    # Remove numbers
  text = re.sub(r"\d+", "", text)

    # Remove punctuation and special characters
  text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
  text = re.sub(r"\s+", "", text).strip()

  return text
  df['text'] = df['text'].apply(clean_text)



In [11]:
df["text"] = df["text"].fillna("")

df = df[df["text"].str.strip() != ""]

df = df.reset_index(drop=True)

In [12]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download("punkt_tab")
nltk.download("stopwords")

df["token"]  = df['text'].apply(word_tokenize)

print(df[["token", "text"]].head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                               token  \
0  [A, security, report, explains, how, hackers, ...   
1  [According, to, a, recent, report, ,, the, hos...   
2  [The, clinic, is, offering, free, health, scre...   
3  [Engineers, released, an, open, source, projec...   
4  [The, retailer, plans, to, expand, its, operat...   

                                                text  
0  A security report explains how hackers can exp...  
1  According to a recent report, the hospital int...  
2  The clinic is offering free health screening t...  
3  Engineers released an open source project to h...  
4  The retailer plans to expand its operations in...  


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
stop_words = set(nltk.corpus.stopwords.words("english"))

In [14]:
df['token_without_stopword'] = df['token'].apply(
    lambda words: [
        word for word in words if word not in stop_words
    ]
)

In [15]:
nltk.download("wordnet")
nltk.download("omw-1.4")
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df["lemitized"] = df['token_without_stopword'].apply(
    lambda words: [
        lemmatizer.lemmatize(word) for word in words
    ]
)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [25]:
df["final_text"] = df["lemitized"].apply(
    lambda words: " ".join(words)
)

print(df[["lemitized", "final_text"]].head(10))

                                           lemitized  \
0  [A, security, report, explains, hacker, exploi...   
1  [According, recent, report, ,, hospital, intro...   
2  [The, clinic, offering, free, health, screenin...   
3  [Engineers, released, open, source, project, h...   
4  [The, retailer, plan, expand, operation, rural...   
5  [The, bank, introduced, service, designed, sma...   
6  [Teachers, adopting, project, based, learning,...   
7  [A, new, study, examined, relationship, physic...   
8  [Doctors, recommend, regular, physical, activi...   
9  [The, hospital, introduced, program, patient, ...   

                                          final_text  
0  A security report explains hacker exploit outd...  
1  According recent report , hospital introduced ...  
2  The clinic offering free health screening memb...  
3  Engineers released open source project help us...  
4    The retailer plan expand operation rural area .  
5  The bank introduced service designed small bus... 

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [26]:
x = df['final_text']
y = df['category']

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)







In [28]:
bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)

X_test_bow = bow.transform(X_test)

print("Training shape:", X_train_bow.shape)
print("Testing shape:", X_test_bow.shape)



Training shape: (320, 382)
Testing shape: (80, 382)


In [29]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)


Training shape: (320, 382)
Testing shape: (80, 382)


In [30]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("y_train length:", len(y_train))

X_train shape: (320,)
y_train shape: (320,)
X_train_tfidf shape: (320, 382)
y_train length: 320


In [31]:
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)

In [32]:
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

In [33]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_test_tfidf)

In [34]:
models = {
    "Naive Bayes": nb_pred,
    "SVM": svm_pred,
    "Logistic Regression": lr_pred
}

for name, prediction in models.items():

    print("\nModel:", name)

    print("Accuracy:",
          accuracy_score(y_test, prediction))

    print("\nClassification Report:")

    print(classification_report(y_test, prediction))


Model: Naive Bayes
Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

     business       1.00      1.00      1.00        10
    education       1.00      1.00      1.00        14
entertainment       1.00      1.00      1.00        12
       health       1.00      1.00      1.00        13
       sports       1.00      1.00      1.00        11
   technology       1.00      1.00      1.00        20

     accuracy                           1.00        80
    macro avg       1.00      1.00      1.00        80
 weighted avg       1.00      1.00      1.00        80


Model: SVM
Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

     business       1.00      1.00      1.00        10
    education       1.00      1.00      1.00        14
entertainment       1.00      1.00      1.00        12
       health       1.00      1.00      1.00        13
       sports       1.00      1.00      1.00        11
   techn

In [35]:
results = []

for name, prediction in models.items():

    accuracy = accuracy_score(y_test, prediction)

    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

result_df = pd.DataFrame(results)

print(result_df.sort_values(
    by="Accuracy",
    ascending=False
))

                 Model  Accuracy
0          Naive Bayes       1.0
1                  SVM       1.0
2  Logistic Regression       1.0


In [43]:
new_review = ["Ram goes to school"]

new_vector = tfidf.transform(new_review)

prediction = svm_model.predict(new_vector)

print("Predicted Sentiment:", prediction[0])

Predicted Sentiment: education


In [37]:
df.head()

,text,category,token,token_without_stopword,lemitized,final_text
0,A security report explains how hackers can exp...,technology,"[A, security, report, explains, how, hackers, ...","[A, security, report, explains, hackers, explo...","[A, security, report, explains, hacker, exploi...",A security report explains hacker exploit outd...
1,"According to a recent report, the hospital int...",health,"[According, to, a, recent, report, ,, the, hos...","[According, recent, report, ,, hospital, intro...","[According, recent, report, ,, hospital, intro...","According recent report , hospital introduced ..."
2,The clinic is offering free health screening t...,health,"[The, clinic, is, offering, free, health, scre...","[The, clinic, offering, free, health, screenin...","[The, clinic, offering, free, health, screenin...",The clinic offering free health screening memb...
3,Engineers released an open source project to h...,technology,"[Engineers, released, an, open, source, projec...","[Engineers, released, open, source, project, h...","[Engineers, released, open, source, project, h...",Engineers released open source project help us...
4,The retailer plans to expand its operations in...,business,"[The, retailer, plans, to, expand, its, operat...","[The, retailer, plans, expand, operations, rur...","[The, retailer, plan, expand, operation, rural...",The retailer plan expand operation rural area .


In [40]:
df['category'].value_counts()

,count
category,
technology,98
education,70
health,66
entertainment,59
sports,55
business,52
